# 토큰과 중간 진행 스트리밍

사용자가 기다리는 동안 토큰, 노드 완료, 업무 진행률을 어떻게 구분해 보여 줄까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 모델이 포함된 그래프


In [2]:
# 모델 호출 전후에 진행률을 내보내는 한 노드 그래프를 구성합니다.
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    answer: str

model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")

def answer(state: State) -> dict[str, str]:
    writer = get_stream_writer()
    writer({"stage": "model_call", "progress": 30})
    response = model.invoke("두 문장으로 답하세요: " + state["question"])
    writer({"stage": "done", "progress": 100})
    return {"answer": response.text}

builder = StateGraph(State)
builder.add_node("answer", answer)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()
print("그래프 준비 완료")


그래프 준비 완료


## 세 종류의 사건 관찰


In [3]:
# 한 실행에서 토큰, 상태 변경분, 사용자 정의 진행률을 함께 셉니다.
seen = {"messages": 0, "updates": 0, "custom": 0}
preview = []
for mode, data in graph.stream(
    {"question": "스트리밍이 체감 지연을 줄이는 이유는 무엇인가요?", "answer": ""},
    stream_mode=["messages", "updates", "custom"],
):
    seen[mode] += 1
    if mode == "messages":
        chunk, metadata = data
        if chunk.text:
            preview.append(chunk.text)
    else:
        print(mode, data)
print("토큰 조각 수:", seen["messages"])
print("답변 앞부분:", "".join(preview)[:80])
assert seen["messages"] > 0 and seen["updates"] > 0 and seen["custom"] == 2


custom {'stage': 'model_call', 'progress': 30}


custom {'stage': 'done', 'progress': 100}
updates {'answer': {'answer': '스트리밍은 데이터를 한꺼번에 다 받지 않고 도착하는 즉시 조금씩 처리해, 사용자가 결과를 더 빨리 보기 시작할 수 있게 해서 체감 지연을 줄입니다. 또한 전송과 처리를 겹쳐 진행하므로 전체 완료 전이라도 첫 응답이 빨리 나타납니다.'}}
토큰 조각 수: 73
답변 앞부분: 스트리밍은 데이터를 한꺼번에 다 받지 않고 도착하는 즉시 조금씩 처리해, 사용자가 결과를 더 빨리 보기 시작할 수 있게 해서 체감 지연을 줄입니


## 표시 정책 만들기


In [4]:
# 사건 종류마다 사용자 화면에 표시할 문구를 구분합니다.
labels = {
    "messages": "답변을 작성하고 있습니다",
    "updates": "단계가 완료되었습니다",
    "custom": "업무 진행률이 바뀌었습니다",
}
for mode in ["messages", "updates", "custom"]:
    print(mode, "→", labels[mode])


messages → 답변을 작성하고 있습니다
updates → 단계가 완료되었습니다
custom → 업무 진행률이 바뀌었습니다
